# Oxford_HIC / Instagram Loading

In [19]:
from string import punctuation

import pandas as pd
from scipy.cluster.hierarchy import single
from sklearn.model_selection import train_test_split

# dirPath = '../Data/Oxford_HIC/CaptionID_oxford_hic_data.csv'
# dirPath = '../Data/Oxford_HIC/Only10_oxford_hic_data.csv'

# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
dirPath = '../Data/Instagram/Generate_mcdonalds_switzerland.csv'

data = pd.read_csv(dirPath)
data['caption'] = data['caption'].str.lower()

# COCO Loading

In [ ]:
### only left 4090 has coco ###
import json
with open('C:/Users/user/fiftyone/coco-2014/raw/captions_train2014.json', 'r') as f:
    data = json.load(f)
data = data['annotations']
imageList = list()
captionList = list()
for item in data:
    image_id = item['image_id']
    caption = item['caption']
    imageList.append(image_id)
    captionList.append(caption)
with open('C:/Users/user/fiftyone/coco-2014/raw/captions_val2014.json', 'r') as f:
    data = json.load(f)
data = data['annotations']
for item in data:
    image_id = item['image_id']
    caption = item['caption']
    imageList.append(image_id)
    captionList.append(caption)

In [ ]:
import pandas as pd
data= pd.DataFrame()
data['image_id'] = imageList
data['caption'] = captionList
data

# Count in dictionary words and out dictionary words

In [20]:
#compute the min, max, mean, and variance of the number of tokens in each caption
import nltk
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer
import torch
inDictSet = set()
outDictSet = set()
tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")

def inDictionary(caption):
    caption = caption.lower()
    split = nltk.word_tokenize(caption)
    inCount = 0
    outCount = 0
    for word in split:
        if word in inDictSet:
            inCount += 1
        elif word in outDictSet:
            outCount += 1
        else:
            token = tokenizer(word, return_tensors="pt").input_ids
            if token.shape[1] == 1:
                inDictSet.add(word)
                inCount += 1
            else:
                outDictSet.add(word)
                outCount += 1
    print(f'inCount: {inCount}, outCount: {outCount}')
    return inCount, outCount

In [21]:
data['inDictCount'], data['outDictCount'] = zip(*data['caption'].map(inDictionary))
print(data.shape)
data.describe()

inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 20, outCount: 6
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 22, outCount: 5
inCount: 22, outCount: 5
inCount: 21, outCount: 5
inCount: 22, outCount: 5
inCount: 22, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 22, outCount: 5
inCount: 22, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5
inCount: 21, outCount: 5


,funny_score,inDictCount,outDictCount
count,226667.000000,226667.000000,226667.000000
mean,0.120684,22.177516,4.672983
std,0.320139,14.372300,5.073316
min,0.000000,0.000000,0.000000
25%,0.000000,14.000000,2.000000
50%,0.000000,20.000000,4.000000
75%,0.000000,24.000000,6.000000
max,1.000000,123.000000,56.000000


get data similar to COCO (inDictCount = 10, inDictCount = 1)

In [24]:
#keep data with outDictCount = 0
new_data = data[data['outDictCount'] <= 3]
print(new_data.shape)
new_data = new_data[new_data['inDictCount'] <= 60]
print(new_data.shape)
new_data.describe()

(105052, 5)
(104962, 5)


,funny_score,inDictCount,outDictCount
count,104962.000000,104962.000000,104962.000000
mean,0.197805,17.078829,1.924868
std,0.390733,6.902728,0.974594
min,0.000000,0.000000,0.000000
25%,0.000000,11.000000,1.000000
50%,0.000000,18.000000,2.000000
75%,0.000000,22.000000,3.000000
max,1.000000,42.000000,3.000000


In [27]:
# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
print("shape of data: ", new_data.shape)
print()
image_id_counts = new_data['image_id'].value_counts()
######################################################################################################
valid_image_ids = image_id_counts[image_id_counts >= 7].index
print("shape of valid_image_ids: ", valid_image_ids.shape)
filtered_data = new_data[new_data['image_id'].isin(valid_image_ids)]
print("shape of filtered_data: ", filtered_data.shape)
train = (
    filtered_data.sort_values(by=['image_id','funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(7)
)
print("shape of train: ", train.shape)
train.to_csv('../Data/Instagram/Only7_simCOCO_mcdonalds_switzerland.csv', index=False)

shape of data:  (104962, 5)

shape of valid_image_ids:  (741,)
shape of filtered_data:  (104843, 5)
shape of train:  (5187, 5)


keep data with outDictCount = 0

In [ ]:
data = data[data['outDictCount'] == 0]
print(data.shape)
data.to_csv('../Data/Oxford_HIC/NoOutDict_oxford_hic_data.csv', index=False)

funny_score top n caption

In [ ]:
top_captions = (
    data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(10)
)
top_captions.shape
top_captions.to_csv('../Data/Oxford_HIC/NoOutDictTop10_oxford_hic_data.csv', index=False)

# Generate data

In [ ]:
from string import punctuation
import pandas as pd
from sklearn.model_selection import train_test_split

# dirPath = '../Data/Oxford_HIC/CaptionID_oxford_hic_data.csv'
# dirPath = '../Data/Oxford_HIC/Only10_oxford_hic_data.csv'

# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
dirPath = '../Data/Instagram/CaptionID_sonicdrivein.csv'

data = pd.read_csv(dirPath)
data['caption'] = data['caption'].str.lower()

In [ ]:
import pandas as pd
import random
from nltk import sent_tokenize
import nltk
from tqdm import tqdm
from albumentations.core.transforms_interface import DualTransform, BasicTransform
import albumentations
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

class NLPTransform(BasicTransform):
    """ Transform for nlp task."""

    @property
    def targets(self):
        return {"data": self.apply}

    def update_params(self, params, **kwargs):
        if hasattr(self, "interpolation"):
            params["interpolation"] = self.interpolation
        if hasattr(self, "fill_value"):
            params["fill_value"] = self.fill_value
        return params

    def get_sentences(self, text, lang='en'):
        return sent_tokenize(text)

In [ ]:
import random
from nltk import sent_tokenize
import nltk
from tqdm import tqdm
from albumentations.core.transforms_interface import DualTransform, BasicTransform
from transformers import BertTokenizer, AutoTokenizer, RobertaTokenizer,BertForMaskedLM, RobertaForMaskedLM, AutoModelForMaskedLM
import albumentations
import torch
import re
import string

class LMmask(NLPTransform):

    def __init__(self, mask_num = 1, tokenizer_name='bert-base-uncased', model_name='bert-base-uncased'):
        self.punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~…“”–’'
        self.mask_num = mask_num
        if tokenizer_name == 'BertTokenizer':
            #   BertTokenizer + BertForMaskedLM  ==> 'bert-base-uncased'
            self.tokenizer = BertTokenizer.from_pretrained(model_name)
            self.model = BertForMaskedLM.from_pretrained(model_name)
            self.mask_token = self.tokenizer.mask_token
            self.mask_token_id = 103
        elif tokenizer_name == 'RobertaTokenizer':
            #   RobertaTokenizer + RobertaForMaskedLM  ==> 'FacebookAI/roberta-base'
            self.tokenizer = RobertaTokenizer.from_pretrained(model_name)
            self.model = RobertaForMaskedLM.from_pretrained(model_name)
            self.mask_token = self.tokenizer.mask_token
            self.mask_token_id = 50264
        elif tokenizer_name == 'AutoTokenizer':
            # AlbertConfig, BartConfig, BertConfig, BigBirdConfig, CamembertConfig, ConvBertConfig, Data2VecTextConfig, DebertaConfig, DebertaV2Config, DistilBertConfig, ElectraConfig, ErnieConfig, EsmConfig, FlaubertConfig, FNetConfig, FunnelConfig, IBertConfig, LayoutLMConfig, LongformerConfig, LukeConfig, MBartConfig, MegaConfig, MegatronBertConfig, MobileBertConfig, MPNetConfig, MraConfig, MvpConfig, NezhaConfig, NystromformerConfig, PerceiverConfig, QDQBertConfig, ReformerConfig, RemBertConfig, RobertaConfig, RobertaPreLayerNormConfig, RoCBertConfig, RoFormerConfig, SqueezeBertConfig, TapasConfig, Wav2Vec2Config, XLMConfig, XLMRobertaConfig, XLMRobertaXLConfig, XmodConfig, YosoConfig.
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForMaskedLM.from_pretrained(model_name)


            self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

    def apply(self, data, top_k=10):
        new_text = []
        # split = nltk.word_tokenize(data)
        split = data.split()
        for idx, n in enumerate(split):
            words = ''
            not_enough = False
            for i in range(self.mask_num):
                if idx + i >= len(split):
                    not_enough = True
                    break
                if split[idx + i] in self.punctuation:
                    words = words + split[idx + i]
                else:
                    words = words + ' ' + split[idx + i]
            if not_enough:
                continue
            words = re.sub(r'\s+', ' ', words)
            words = words.strip()
            token = self.tokenizer.encode(words, return_tensors="pt")
            single = token.shape[1] == self.mask_num + 2
            if single:
                text = ''
                # words = nltk.word_tokenize(words)
                words = words.split()
                for idx_w, word_split in enumerate(split):
                    if word_split in words and idx_w in range(idx, idx + self.mask_num):
                        words.remove(word_split)
                        text = text + ' ' + self.mask_token
                    elif word_split in self.punctuation:
                        text = text + word_split
                    else:
                        text = text + ' ' + word_split
                text = re.sub(r'\s+', ' ', text)
                text = text.strip()
                # DEFINE SENTENCE
                indices = self.tokenizer.encode(text, add_special_tokens=True, return_tensors='pt')
                # PREDICT MISSING WORDS
                pred = self.model(indices)
                masked_indices = torch.where(indices == self.mask_token_id)[1]
                # TOP 10 PREDICTIONS
                top10 = torch.topk(pred[0][0][masked_indices, :], top_k, axis=1)
                # FILL IN MISSING WORDS
                for i in range(top_k):
                    temp = text
                    for j in range(self.mask_num):
                        if j < top10.indices.shape[0] and i < top10.indices.shape[1]:
                            temp = temp.replace(self.mask_token, self.tokenizer.decode(top10.indices[j][i]), 1)
                    temp = re.sub(r'\s+', ' ', temp)
                    new_text.append(temp)
        return new_text

# Generate data not in dictionary

In [32]:
from string import punctuation
import pandas as pd
from sklearn.model_selection import train_test_split

# dirPath = '../Data/Oxford_HIC/CaptionID_oxford_hic_data.csv'
# dirPath = '../Data/Oxford_HIC/Only10_oxford_hic_data.csv'

# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
dirPath = '../Data/Instagram/CaptionID_wendys.csv'

data = pd.read_csv(dirPath)

In [33]:
# lower caption
data['caption'] = data['caption'].str.lower()

In [34]:
import pandas as pd
import random
from nltk import sent_tokenize
import nltk
from tqdm import tqdm
from albumentations.core.transforms_interface import DualTransform, BasicTransform
import albumentations
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')

class NLPTransform(BasicTransform):
    """ Transform for nlp task."""

    @property
    def targets(self):
        return {"data": self.apply}

    def update_params(self, params, **kwargs):
        if hasattr(self, "interpolation"):
            params["interpolation"] = self.interpolation
        if hasattr(self, "fill_value"):
            params["fill_value"] = self.fill_value
        return params

    def get_sentences(self, text, lang='en'):
        return sent_tokenize(text)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\TonyLab\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\TonyLab\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [58]:
import random
from nltk import sent_tokenize
import nltk
from tqdm import tqdm
from albumentations.core.transforms_interface import DualTransform, BasicTransform
from transformers import BertTokenizer, AutoTokenizer, RobertaTokenizer,BertForMaskedLM, RobertaForMaskedLM, AutoModelForMaskedLM
import albumentations
import torch
import re
import string

class LMmask(NLPTransform):

    def __init__(self, mask_num = 1, tokenizer_name='bert-base-uncased', model_name='bert-base-uncased'):
        self.punctuation = '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~…“”–’'
        self.mask_num = mask_num
        if tokenizer_name == 'BertTokenizer':
            #   BertTokenizer + BertForMaskedLM  ==> 'bert-base-uncased'
            self.tokenizer = BertTokenizer.from_pretrained(model_name)
            self.model = BertForMaskedLM.from_pretrained(model_name)
            self.mask_token = self.tokenizer.mask_token
            self.mask_token_id = 103
        elif tokenizer_name == 'RobertaTokenizer':
            #   RobertaTokenizer + RobertaForMaskedLM  ==> 'FacebookAI/roberta-base'
            self.tokenizer = RobertaTokenizer.from_pretrained(model_name)
            self.model = RobertaForMaskedLM.from_pretrained(model_name)
            self.mask_token = self.tokenizer.mask_token
            self.mask_token_id = 50264
        elif tokenizer_name == 'AutoTokenizer':
            # AlbertConfig, BartConfig, BertConfig, BigBirdConfig, CamembertConfig, ConvBertConfig, Data2VecTextConfig, DebertaConfig, DebertaV2Config, DistilBertConfig, ElectraConfig, ErnieConfig, EsmConfig, FlaubertConfig, FNetConfig, FunnelConfig, IBertConfig, LayoutLMConfig, LongformerConfig, LukeConfig, MBartConfig, MegaConfig, MegatronBertConfig, MobileBertConfig, MPNetConfig, MraConfig, MvpConfig, NezhaConfig, NystromformerConfig, PerceiverConfig, QDQBertConfig, ReformerConfig, RemBertConfig, RobertaConfig, RobertaPreLayerNormConfig, RoCBertConfig, RoFormerConfig, SqueezeBertConfig, TapasConfig, Wav2Vec2Config, XLMConfig, XLMRobertaConfig, XLMRobertaXLConfig, XmodConfig, YosoConfig.
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForMaskedLM.from_pretrained(model_name)


            self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

    def apply(self, data, top_k=10):
        new_text = []
        split = data.split()
        def generateINDICT(split, idx, former):
            n = split[idx]
            token = self.tokenizer.encode(n, return_tensors="pt")
            single = token.shape[1] == 3

            if not single:
                words = data.replace(n, self.mask_token, 1)
                # DEFINE SENTENCE
                indices = self.tokenizer.encode(words, add_special_tokens=True, return_tensors='pt')
                # PREDICT MISSING WORDS
                pred = self.model(indices)
                masked_indices = torch.where(indices == self.mask_token_id)[1]
                # TOP 10 PREDICTIONS
                top10 = torch.topk(pred[0][0][masked_indices, :], top_k, axis=1)
                # FILL IN MISSING WORDS
                for i in range(top_k):
                    temp = words
                    for j in range(self.mask_num):
                        if j < top10.indices.shape[0] and i < top10.indices.shape[1]:
                            temp = temp.replace(self.mask_token, self.tokenizer.decode(top10.indices[j][i]), 1)
                    temp = re.sub(r'\s+', ' ', temp)
                    if idx + 1 < len(split):
                        generateINDICT(split, idx + 1, former)
                    else:
                        new_text.append(temp)
            else:
                if idx + 1 < len(split):
                    generateINDICT(split, idx + 1, former)
                else:
                    new_text.append(former)
        generateINDICT(split, 0, data)
        return new_text

In [94]:
words_not_in_dict = []
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
for i in range(data.shape[0]):
    text = data.caption[i]
    # remove punctuation
    text = text.translate(str.maketrans('', '', '!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~…“”–’'))
    #lowercase
    text = text.lower()
    split = text.split()
    for word in split:
        single = tokenizer.encode(word, return_tensors="pt").shape[1] == 3
        # check if word is number
        number = False
        try:
            int(word)
            number = True
        except:
            pass
        if not single and not number:
            words_not_in_dict.append(word)

In [95]:
words_not_in_dict = set(words_not_in_dict)
print(len(words_not_in_dict))
words_not_in_dict

201


{'2pm',
 '3breakfastdeal',
 '9am',
 'affirmation',
 'ahhthe',
 'alchemy',
 'altruistic',
 'amiright',
 'applewood',
 'arent',
 'asmr',
 'aunties',
 'baconator',
 'baconholder',
 'bby',
 'beefmas',
 'beefy',
 'bfast',
 'bff',
 'biggie',
 'bogo',
 'boomers',
 'bouba',
 'brandedcontent',
 'bravoandy',
 'bravocon',
 'bravocon2023',
 'bravotv',
 'breakfastfortniteflowers',
 'bruh',
 'burgerverse',
 'cant',
 'capes',
 'carterrthompson',
 'carton',
 'cemented',
 'charcuter',
 'cheugy',
 'chillin',
 'chills',
 'chivalry',
 'cinnabon',
 'clueless',
 'crave',
 'crispy',
 'cupholder',
 'dangot',
 'delete',
 'devours',
 'diff',
 'dont',
 'doordash',
 'dreaminterpretationscom',
 'drivethru',
 'dudes',
 'eiffry',
 'elses',
 'emoji',
 'enlightened',
 'everyones',
 'fav',
 'feastoflegends',
 'feastoflegendscom',
 'feasts',
 'feelin',
 'flexin',
 'foodart',
 'freshmade',
 'fringer',
 'frosty',
 'frostyll',
 'frostys',
 'gettin',
 'giphy',
 'glish',
 'godzillatoho',
 'goto',
 'grad',
 'grubif',
 'hed',


In [105]:
temp = tokenizer.encode('lol', return_tensors="pt")
for i in range(temp.shape[1]):
    print(f'{i}: {tokenizer.decode(temp[0][i])}')
# tokenizer.decode(101)

0: [CLS]
1: lo
2: ##l
3: [SEP]


In [61]:
# complete following code
new_data = pd.DataFrame(columns=['caption', 'image_id', 'funny_score'])

with tqdm(total=data.shape[0]) as progress_bar:
    for i in range(data.shape[0]):
        text = data.caption[i]
        image_id = data.image_id[i]
        funnyscore = data.funny_score[i]
        new_data = pd.concat([new_data, pd.DataFrame([[text, image_id, funnyscore]], columns=['caption', 'image_id', 'funny_score'])], ignore_index=True)
        generate_wordcount = len(nltk.word_tokenize(text)) - 3
        # print(f'image_id{image_id}, funnyscore{funnyscore}, generate_wordcount{generate_wordcount}, text{text}')
        if generate_wordcount > 0:
            for j in range(min(generate_wordcount, 1)):
                # lm = LMmask(mask_num = i+1, tokenizer_name='BertTokenizer', model_name='bert-base-uncased')
                lm = LMmask(mask_num = j+1, tokenizer_name='RobertaTokenizer', model_name='FacebookAI/roberta-base')
                temp_sentences = lm.apply(text, top_k=5)

                for sentence in temp_sentences:
                    new_data = pd.concat([new_data, pd.DataFrame([[sentence, image_id, funnyscore]], columns=['caption', 'image_id', 'funny_score'])], ignore_index=True)
                progress_bar.set_postfix({"j": j, "k": len(temp_sentences), "size": new_data.shape})
        progress_bar.update()
new_data['caption'] = new_data['caption'].str.lower()
# remove duplicate data
new_data = new_data.drop_duplicates()

  0%|          | 0/367 [00:00<?, ?it/s]C:\Users\TonyLab\AppData\Local\Temp\ipykernel_9764\3314192904.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  new_data = pd.concat([new_data, pd.DataFrame([[text, image_id, funnyscore]], columns=['caption', 'image_id', 'funny_score'])], ignore_index=True)
  0%|          | 1/367 [00:01<08:01,  1.31s/it, j=0, k=0, size=(1, 3)]

words: saucy, token: tensor([[   0,   29, 1180, 4469,    2]])
words: nuggs, token: tensor([[    0,   282, 15106,    29,     2]])
words: you’ll, token: tensor([[   0, 6968,   17,   27,  890,    2]])
words: yourself, token: tensor([[    0, 16625, 13367,     2]])
words: 🔥🔥🔥🤣🏈, token: tensor([[    0,  6569, 10674,  8210,  6569, 10674,  8210,  6569, 10674,  8210,
          6569, 10470,  2469,  6569,  9357, 23133,     2]])
words: wendys, token: tensor([[   0,  605, 1397, 2459,    2]])
words: flavors, token: tensor([[   0, 4825, 1469,  994,    2]])
words: don’t, token: tensor([[   0, 7254,   17,   27,   90,    2]])
words: one’s, token: tensor([[   0, 1264,   17,   27,   29,    2]])


  1%|          | 2/367 [00:02<07:21,  1.21s/it, j=0, k=0, size=(2, 3)]

words: can’t, token: tensor([[   0, 7424,   17,   27,   90,    2]])
words: believe, token: tensor([[    0,  8494, 16637,     2]])
words: saucy, token: tensor([[   0,   29, 1180, 4469,    2]])
words: nugg, token: tensor([[    0,   282, 15106,     2]])
words: legacy, token: tensor([[   0, 4308, 5073,    2]])
words: forever, token: tensor([[    0, 13364,  2802,     2]])
words: cemented, token: tensor([[    0,  1755, 19601,     2]])
words: canvas, token: tensor([[    0,  7424, 20670,     2]])
words: 2024, token: tensor([[   0,  844, 1978,    2]])


  1%|          | 3/367 [00:03<07:17,  1.20s/it, j=0, k=0, size=(3, 3)]

...

  3%|▎         | 11/367 [00:11<07:45,  1.31s/it, j=0, k=0, size=(11, 3)]

words: lonely, token: tensor([[   0,  462, 1264,  352,    2]])
words: lonely, token: tensor([[   0,  462, 1264,  352,    2]])
words: wendys, token: tensor([[   0,  605, 1397, 2459,    2]])
words: 073124, token: tensor([[    0,  3570,   246, 22960,     2]])
words: 9am, token: tensor([[  0, 466, 424,   2]])
words: mst, token: tensor([[  0, 119, 620,   2]])
words: participating, token: tensor([[    0, 42038,  1295,     2]])
words: wendy’s®, token: tensor([[   0,  605, 1397,  219,   17,   27,   29, 8716,    2]])
words: 073124080724, token: tensor([[    0,  3570, 29930,  1749, 38633,  1978,     2]])
words: spicy, token: tensor([[    0,  4182, 24363,     2]])
words: pepper, token: tensor([[   0, 2379, 5961,    2]])
words: nuggs, token: tensor([[    0,   282, 15106,    29,     2]])
words: pepper, token: tensor([[   0, 2379, 5961,    2]])
words: dipping, token: tensor([[    0,   417, 11813,     2]])
words: sauce, token: tensor([[   0,   29, 1180, 1755,    2]])
words: fry, token: tensor([[   0,

  4%|▎         | 13/367 [00:13<06:44,  1.14s/it, j=0, k=0, size=(13, 3)]

words: that’s, token: tensor([[   0, 6025,   17,   27,   29,    2]])


  4%|▍         | 15/367 [00:16<06:16,  1.07s/it, j=0, k=0, size=(14, 3)]

KeyboardInterrupt



In [ ]:
new_data

In [10]:
# 計算每個 image_id 的資料數量
image_id_counts = new_data['image_id'].value_counts()
print(f'Number of unique image_id: {len(image_id_counts)}')
# 篩選出有 1000 條以上資料的 image_id
valid_image_ids = image_id_counts[image_id_counts >= 300].index
print(f'Number of image_id with 300 captions: {len(valid_image_ids)}')
# 篩選原始資料
filtered_data = new_data[new_data['image_id'].isin(valid_image_ids)]
print(f'Number of data: {filtered_data.shape[0]}')
top_captions = (
    filtered_data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(300)
)
print(f'Number of data: {top_captions.shape[0]}')

Number of unique image_id: 1125
Number of image_id with 300 captions: 262
Number of data: 110815
Number of data: 78600


In [ ]:
# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
# mcdonalds_switzerland images: 262/1125 , data = 78600/110815/226667
print(new_data.shape)
new_data.to_csv('../Data/Instagram/Generate_mcdonalds_switzerland.csv', index=False)# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
# mcdonalds_switzerland images: 262/1125 , data = 78600/110815/226667
print(new_data.shape)
new_data.to_csv('../Data/Instagram/Generate_mcdonalds_switzerland.csv', index=False)

# Check distribution of how many captions in each image

In [3]:
import pandas as pd
# ff_list=['mcdonalds', 'mcdonalds_switzerland', 'mcdonaldscanada', 'sonicdrivein','wendys']
name = 'wendys'
fileName = 'Generate_' + name
dirPath = '../Data/Instagram/' + fileName + '.csv'
data = pd.read_csv(dirPath)
print("shape of data: ", data.shape)
print()
image_id_counts = data['image_id'].value_counts()
######################################################################################################
print(f'          {fileName}')
print(f'           Total image counts: {len(image_id_counts)}')
x = image_id_counts[image_id_counts >= 1000]
sum = len(x)
print(f' {1000} <= caption        --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 300]
# sum = len(x)
# print(f' {300} <= caption        --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 200]
# x = x[x < 300]
# sum += len(x)
# print(f'{200:4d} <= caption < {300:4d} --- {len(x):6d} --- {sum:6d}')
# x = image_id_counts[image_id_counts >= 100]
# x = x[x < 200]
# sum += len(x)
# print(f'{100:4d} <= caption < {200:4d} --- {len(x):6d} --- {sum:6d}')
for i in range(10):
    x = image_id_counts[image_id_counts >= (10-1-i)*100]
    x = x[x < (10-i)*100]
    sum += len(x)
    print(f' {(10-1-i)*100:4d} <= caption < {(10-i)*100:4d} --- {len(x):6d} --- {sum:6d}')


shape of data:  (35727, 3)

          Generate_wendys
           Total image counts: 367
 1000 <= caption        ---      0 ---      0
  900 <= caption < 1000 ---      0 ---      0
  800 <= caption <  900 ---      0 ---      0
  700 <= caption <  800 ---      1 ---      1
  600 <= caption <  700 ---      0 ---      1
  500 <= caption <  600 ---      1 ---      2
  400 <= caption <  500 ---      4 ---      6
  300 <= caption <  400 ---      8 ---     14
  200 <= caption <  300 ---     33 ---     47
  100 <= caption <  200 ---     91 ---    138
    0 <= caption <  100 ---    229 ---    367


#########################################

In [ ]:
text = "You can’t achieve strawberry lemonade until you first get strawberry lemonade followed by strawberry lemonade."
new_data = pd.DataFrame()
for i in range(3):
    # lm = LMmask(mask_num = i+1, tokenizer_name='BertTokenizer', model_name='bert-base-uncased')
    lm = LMmask(mask_num = i+1, tokenizer_name='RobertaTokenizer', model_name='FacebookAI/roberta-base')
# lm = LMmask(mask_num = 1, tokenizer_name='AutoTokenizer', model_name='')
    # 'DataFrame' object has no attribute 'append'

    new_data = pd.concat([new_data, pd.DataFrame(lm.apply(text, top_k=5))], ignore_index=True)
    new_data = pd.concat([new_data, pd.DataFrame(["============================================================="])], ignore_index=True)
new_data
# lm = LMmask(mask_num = 1, tokenizer_name='BertTokenizer', model_name='bert-base-uncased')
# lm.apply(text, top_k=5)


In [ ]:
new_data.to_csv('./roberta2.csv', index=False)

In [ ]:
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")
tokenizer.decode(2226)


In [ ]:
import string
from torch.utils.data import Dataset, DataLoader
import clip
import os
import pandas as pd
import pickle
from torch import nn
import numpy as np
import torch
import torch.nn.functional as nnf
import sys
from typing import Tuple, List, Union, Optional
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    AdamW,
    get_linear_schedule_with_warmup,
)
from transformers import AutoConfig, AutoTokenizer, Gemma2ForCausalLM
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model
import PIL.Image
from tqdm import tqdm
tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")
inwordList = set()
outwordList = set()
progress_counter = tqdm(total=len(data), desc='Counting tokens', position=0, leave=True)
progress_inDict = tqdm(total=len(data), desc='Counting in dict', position=0, leave=True)
def token_counter(caption):
    caption = caption.lower()
    caption = caption.translate(str.maketrans('', '', string.punctuation))
    words = caption.split()
    temp = set(words)
    progress_counter.update(1)
    return len(temp)

def word_in_dict(caption):
    # caption = caption.lower()
    caption = caption.translate(str.maketrans('', '', string.punctuation))
    words = caption.split()
    temp = set(words)
    notTested = temp.difference(inwordList).difference(outwordList)
    notInDictCounter = len(temp.intersection(outwordList))
    # print(f'notInDictCounter: {notInDictCounter}, notTested: {notTested}')
    for word in notTested:
        token = tokenizer(word, return_tensors="pt").input_ids
        # print(f'word: {word}, token: {token}')
        if token.shape[1] == 1:
            inwordList.add(word)
        else:
            notInDictCounter += 1
            outwordList.add(word)
    # progress_inDict.update(1)
    return notInDictCounter

In [ ]:
train, test = train_test_split(data, test_size=0.2, random_state=42)

In [ ]:
# 計算每個 caption 的字數
data['token_count'] = data['caption'].apply(token_counter)
progress_counter.close()
# 計算每個 caption 中不在字典內的字數
data['out_of_dict_count'] = data['caption'].apply(word_in_dict)
progress_inDict.close()
print(f'Number of unique words in the dictionary: {len(inwordList)}')
print(f'Number of unique words out of the dictionary: {len(outwordList)}')

In [ ]:
test_tokens = inwordList.union(outwordList)

In [ ]:
train_tokens = inwordList.union(outwordList)

In [ ]:
a = set(test_tokens)
b = set(train_tokens)
c = set()
d = set()
for testDatasetToken in a:
    c.add(testDatasetToken.item())
print(len(c))
for testDatasetToken in b:
    d.add(testDatasetToken.item())
print(len(d))
print(len(c.symmetric_difference(d)))
print(len(c.intersection(d)))

In [ ]:
print(data.shape)
print(data[data['out_of_dict_count'] < 4].shape)
print(data[data['out_of_dict_count'] < 3].shape)
print(data[data['out_of_dict_count'] < 2].shape)
print(data[data['out_of_dict_count'] < 1].shape)
print(data[data['out_of_dict_count'] < 0].shape)
data.describe()

In [ ]:
data = data[data['out_of_dict_count'] < 1]

In [ ]:
# 計算每個 image_id 的資料數量
image_id_counts = data['image_id'].value_counts()
# 篩選出有 1000 條以上資料的 image_id
valid_image_ids = image_id_counts[image_id_counts >= 1000].index
# 篩選原始資料
filtered_data = data[data['image_id'].isin(valid_image_ids)]
print(len(valid_image_ids))
top_captions = (
    filtered_data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(1000)
)
print(top_captions.shape)
print(filtered_data.shape)

In [ ]:
print(f'          Total image counts: {len(image_id_counts)}')
x = image_id_counts[image_id_counts >= 1000]
sum = len(x)
print(f' {1000} <= caption        --- {len(x):6d} --- {sum:6d}')
for i in range(10):

    if i == 9:
        x = image_id_counts[image_id_counts >= 50]
        x = x[x < (10-i)*100]
        sum += len(x)
        print(f' {50:4d} <= caption < {(10-i)*100:4d} --- {len(x):6d} --- {sum:6d}')
        x = image_id_counts[image_id_counts >= 10]
        x = x[x < 50]
        sum += len(x)
        print(f' {10:4d} <= caption < {50:4d} --- {len(x):6d} --- {sum:6d}')
        x = image_id_counts[image_id_counts < 10]
        sum += len(x)
        print(f' {(10-1-i)*100:4d} <= caption < {10:4d} --- {len(x):6d} --- {sum:6d}')
    else:
        x = image_id_counts[image_id_counts >= (10-1-i)*100]
        x = x[x < (10-i)*100]
        sum += len(x)
        print(f' {(10-1-i)*100:4d} <= caption < {(10-i)*100:4d} --- {len(x):6d} --- {sum:6d}')


In [ ]:
# get data differ from data_all and data1500
print(data.shape)
print(filtered_data.shape)
exceptdata = data[~data['image_id'].isin(filtered_data['image_id'])]
print(len(exceptdata['image_id'].value_counts()))
print(exceptdata.shape)
exceptdata.to_csv('../Data/Oxford_HIC/except1000up_oxford_hic_data.csv', index=False)

In [ ]:
filtered_data.to_csv('../Data/Oxford_HIC/1000up_oxford_hic_data.csv', index=False)

In [ ]:
top_captions = (
    filtered_data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(1)
)
top_captions.shape
top_captions.to_csv('../Data/Oxford_HIC/Only10_oxford_hic_data.csv', index=False)

In [ ]:
top_captions.to_csv('../Data/Oxford_HIC/Only10_oxford_hic_data.csv', index=False)

In [ ]:
whole = pd.read_csv('../Data/Oxford_HIC/Only10_oxford_hic_data.csv')
unique_image_ids = whole['image_id'].unique()

train_ids, test_ids = train_test_split(unique_image_ids, test_size=0.2, random_state=42)
train = whole[whole['image_id'].isin(train_ids)]
test = whole[whole['image_id'].isin(test_ids)]
print(train.shape, test.shape)

In [ ]:
train_mess = pd.DataFrame()
test_mess = pd.DataFrame()
for image_id, group in whole.groupby("image_id"):
    train_split, test_split = train_test_split(group, test_size=0.2, random_state=42)
    train_mess = pd.concat([train_mess, train_split])
    test_mess = pd.concat([test_mess, test_split])
print(f'train: {train_mess.shape}')
print(f'test: {test_mess.shape}')

In [ ]:
train_image = ['2spbgym', 'all-the-things', 'imgflip_0', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_504']
train_data = train[train['image_id'].isin(train_image)]
test_image = ['bokete_100174', 'imgflip_834']
test_data = test[test['image_id'].isin(test_image)]
print(train_data.shape, test_data.shape)
train_data.to_csv('../Data/Oxford_HIC/Train_Only10_oxford_hic_data.csv', index=False)
test_data.to_csv('../Data/Oxford_HIC/Test_Only10_oxford_hic_data.csv', index=False)

In [ ]:
train_image = ['2spbgym', 'all-the-things', 'imgflip_0', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_504','bokete_100174', 'imgflip_834']
train_data_mess = train_mess[train_mess['image_id'].isin(train_image)]
test_image =  ['2spbgym', 'all-the-things', 'imgflip_0', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_504','bokete_100174', 'imgflip_834']
test_data_mess = test_mess[test_mess['image_id'].isin(test_image)]
print(train_data_mess.shape, test_data_mess.shape)
train_data.to_csv('../Data/Oxford_HIC/Train_Only10_mess_oxford_hic_data.csv', index=False)
test_data.to_csv('../Data/Oxford_HIC/Test_Only10_mess_oxford_hic_data.csv', index=False)

In [ ]:
train_data_mess

In [ ]:
# list same caption in train_data and test_data_mess
a = test_data[test_data['image_id'] == 'imgflip_130']
b = test_data_mess[test_data_mess['image_id'] == 'imgflip_130']
x = set(a['caption']).intersection(set(b['caption']))
for i in x:
    if '' in i:
        print(i)
# 2spbgym,climb a mountain? pff, i have wings...
# all-the-things,go to a pizza buffet eat all the pizza
# imgflip_0,12 dollars; 11 dollars with 1 dollar shipping
# imgflip_1033,I POUR MILK BEFORE CEREAL
# imgflip_11,ME WAITING FOR MY INTERNET TO RECONNECT
# imgflip_117,calling the teacher mom
# imgflip_16,IF SOMEONE DIES IN THE LIVING ROOM... IS IT STILL CALLED THE LIVING ROOM?
# imgflip_189,you; losing a few seconds of your life looking at this
# imgflip_23,me: gets up and starts clapping because the chiefs won; the guy who has been pushing my wheelchair for 10 years
# imgflip_504,THEY TOOK AWAY MY HAPPY MEAL I TOOK AWAY THEIR HAPPINESS

# i-love-coloring-kid,she started writing notes !!
# imgflip_130,WHEN YOUR FRIEND; DOSENT LIKE ROOT BEER

In [ ]:
a

In [ ]:
img_names = data['image_id'].unique()
len(img_names)

In [ ]:
top_captions = (
    data.sort_values(by=['image_id', 'funny_score'], ascending=[True, False])
    .groupby('image_id')
    .head(50)
)
top_captions.shape

In [ ]:
train_data = pd.DataFrame()
test_data = pd.DataFrame()
only = pd.DataFrame()
for image_id, group in top_captions.groupby("image_id"):
    if group.shape[0] < 50:
        continue
    only = pd.concat([only, group])
    # train, test = train_test_split(group, test_size=0.2, random_state=42)
    # train_data = pd.concat([train_data, train])
    # test_data = pd.concat([test_data, test])
print(only.shape)

In [ ]:
only.to_csv('../Data/Oxford_HIC/Only50_oxford_hic_data.csv', index=False)

In [ ]:
top_captions.to_csv('../Data/Oxford_HIC/Top10_oxford_hic_data.csv', index=False)

In [ ]:
# 獲取唯一的 image_id
unique_image_ids = top_captions['image_id'].unique()
print(unique_image_ids.shape)
# 將 image_id 拆分為 80% 訓練集和 20% 測試集
train_ids, test_ids = train_test_split(unique_image_ids, test_size=0.2, random_state=42)

# 根據拆分的 image_id 選取資料
train_data = top_captions[top_captions['image_id'].isin(train_ids)]
test_data = top_captions[top_captions['image_id'].isin(test_ids)]
print(train_data.shape, test_data.shape)

In [ ]:
unique_image_ids.shape[0]/3

In [ ]:
a = unique_image_ids[:30000]
print(a.shape)

In [ ]:
dirPath = '../Data/Oxford_HIC/Only1200_oxford_hic_data.csv'
data = pd.read_csv(dirPath)
print("shape of data: ", data.shape)
######################################################################################################
train = pd.DataFrame()
test = pd.DataFrame()
for image_id, group in data.groupby("image_id"):
    train_split, test_split = train_test_split(group, test_size=0.2, random_state=42)
    train = pd.concat([train, train_split])
    test = pd.concat([test, test_split])
print(f'train: {train.shape}')
print(f'test: {test.shape}')
######################################################################################################
# unique_image_ids = data['image_id'].unique()
# # unique_image_ids = unique_image_ids[:30000]
# # unique_image_ids, rest = train_test_split(unique_image_ids, test_size=0.7, random_state=42)
# # print(unique_image_ids.shape)
# train_ids, test_ids = train_test_split(unique_image_ids, test_size=0.2, random_state=42)
# train = data[data['image_id'].isin(train_ids)]
# test = data[data['image_id'].isin(test_ids)]
# print(train.shape, test.shape)

In [ ]:
train=train.reset_index()
test=test.reset_index()

In [ ]:



train_image = ['imgflip_0', 'imgflip_101', 'imgflip_1033','imgflip_11', 'imgflip_117', 'imgflip_16', 'imgflip_189','imgflip_23', 'imgflip_47', 'imgflip_504']
train_text = ['12 dollars; 11 dollars with 1 dollar shipping'
              ,'I HAD A GIRLFRIEND; AAAAAAND ITS GONE'
              ,'I POUR THE CEREAL AFTER I POUR THE MILK'
              ,'WAITING FOR MY PHONE TO GET  TO 100%'
              ,'You when you have over one test at school in a day'
              ,'IF SOMEONE WANTS TO KILL YOU; GO TO A LIVING ROOM'
              ,'you; eating 5 pounds of cheese; every day; your stomach'
              ,'Me:stands up to stretch my legs; The person who had been pushing my wheelchair for the last 26 years'
              ,'Me: Opens door for some fresh air; Everyone else in the submarine:'
              ,'THEY TOOK AWAY MY HAPPY MEAL I TOOK AWAY THEIR HAPPINESS']
# imgflip_130,WHEN YOU SEE PICS OF YOUR FRIENDS HANGING OUT; BUT YOU WEREN'T INVITED
# imgflip_659,When the teacher uses your voice recording on the homework as an example
test_image = ['imgflip_130', 'imgflip_659']
test_text = ['0 VIEWS 5 DISLIKES'
              ,'when the mobile game ad is so laggy that it crashes your game and you lose out on a reward:']

tokens_list = []
mask_list = []
prefix_list = []
train_gt = []
train_caption = dict()
train_image_id_list = []
# mess = set()
mess_a = set()
mess_b =set()
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("tiiuae/Falcon3-1B-Base")
print(train.shape)
for i in range(train.shape[0]):
    caption = train['caption'][i]
    image_id = train['image_id'][i]
    # if image_id == 'imgflip_117':#and 'GONE' in caption:
    #     notmess.add(caption)
    #     print(f"Image ID: {image_id}, Caption: {caption}")
# train_image = ['', 'imgflip_101', '','', 'imgflip_117', '', '','', '', '']


    if image_id in train_image and caption in train_text:
        print(f"Image ID: {image_id}, Caption: {caption}")
print("===================================================================================")
for i in range(test.shape[0]):
    caption = test['caption'][i]
    image_id = test['image_id'][i]
    if image_id == 'imgflip_130':#and 'GONE' in caption:
        mess_a.add(caption)
    if image_id == 'imgflip_659':
        mess_b.add(caption)
    if image_id in test_image and caption in test_text:
        print(f"Image ID: {image_id}, Caption: {caption}")


In [ ]:
set.intersection(notmess_b, mess_b)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-1.3B")
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-2.7B")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-1.3B")
model = AutoModelForCausalLM.from_pretrained("EleutherAI/gpt-neo-2.7B")

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B")